In [ ]:
# 1. Instalação
# pip install -q bertopic[visualization] sentence-transformers umap-learn hdbscan
# pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
 2. Importações
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

from umap import UMAP
import hdbscan

In [ ]:
# 3. Verificar GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
#Esperado ==> Device: cuda

In [ ]:
#4. Carregar dados
df = pd.read_csv("reclamacoes.csv")

# supondo coluna chamada 'texto'
docs = df['texto'].dropna().tolist()

len(docs)

In [ ]:
# 5. (Opcional) Limpeza leve
import re

def limpar(texto):
    texto = texto.lower()
    texto = re.sub(r'\d+', '', texto)
    texto = re.sub(r'[^\w\s]', '', texto)
    return texto

docs = [limpar(t) for t in docs]

In [ ]:
# 6. Embeddings na GPU
model = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2',
    device=device
)

embeddings = model.encode(
    docs,
    batch_size=64,          # ajuste conforme VRAM
    show_progress_bar=True
)

In [ ]:
# 7. Redução de dimensionalidade (UMAP)
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine'
)

In [ ]:
# 8. Clusterização (HDBSCAN)
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=30,
    metric='euclidean',
    cluster_selection_method='eom'
)

In [ ]:
# 9. Modelo BERTopic
topics, probs = topic_model.fit_transform(docs, embeddings)

In [ ]:
# 10. Treinar modelo
topic_info = topic_model.get_topic_info()
topic_info.head(10)

In [ ]:
# 11. Ver os principais temas
topic_info = topic_model.get_topic_info()
topic_info.head(10)

In [ ]:
# 12. Ver palavras de um tópico
topic_model.get_topic(0)

In [ ]:
# 13. Ver exemplos de reclamações por tema
topic_model.get_representative_docs(0)

In [ ]:
# 14. Visualização (muito útil)
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart()

In [ ]:
#15. Nomear temas com LLM (opcional)
topic_model.update_topics(docs)

In [ ]:
# Otimizações para RTX 3060
# Ajustes recomendados:
embeddings = model.encode(
    docs,
    batch_size=64,   # aumente se couber na VRAM
    normalize_embeddings=True
)

In [ ]:
# Resultado esperado

# Você vai obter clusters como:

# “atraso entrega”
# “cobrança indevida”
# “cancelamento difícil”
# “produto com defeito”

# Cada um com:

# quantidade de ocorrências
# palavras-chave
# exemplos reais

In [ ]:
# Extensão prática (produção)

# Depois disso, você pode:

# Criar dashboard (Power BI / Streamlit)
# Monitorar evolução de temas no tempo
# Detectar novos problemas automaticamente